In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('nitrous_oxide_production')

In [4]:
bd.databases

Databases dictionary with 10 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050
	nitrous-oxide-ei310-Base-2020
	nitrous-oxide-ei310-Base-2050
	nitrous-oxide-ei310-RCP19-2050
	nitrous-oxide-ei310-RCP26-2050

In [5]:
base_db = wurst.extract_brightway2_databases('nitrous-oxide-ei310-Base-2020')

Getting activity data


100%|██████████| 21/21 [00:00<?, ?it/s]


Adding exchange data to activities


100%|██████████| 191/191 [00:00<00:00, 17403.37it/s]


Filling out exchange data


100%|██████████| 21/21 [00:00<00:00, 584.89it/s]


In [6]:
database_names = bd.databases
ecoinvent_db_names = []
premise_db_names = []
for db_name in database_names:
    if ('Base' in db_name or 'RCP' in db_name) and 'nitrous-oxide' not in db_name and 'Base-2020' not in db_name:
        ecoinvent_db_names.append(db_name)
        premise_db_names.append(db_name.replace('ecoinvent-3.10-cutoff-', ''))
ecoinvent_db_names.sort()
premise_db_names.sort()

premise_db_names

['Base-2050', 'RCP19-2050', 'RCP26-2050']

In [7]:
biosphere_db = bd.Database('ecoinvent-3.10-biosphere') # import the biosphere database
biosphere_db_dict = [ds.as_dict() for ds in biosphere_db] # convert biosphere database to dictionary # maybe not needed?

In [8]:
for premise_db_name in premise_db_names:
    print('linking database ' + premise_db_name + '...')
    premise_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-cutoff-' + premise_db_name)]

    db_linked = copy.deepcopy(base_db)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in db_linked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchange_link = wurst.get_one(base_db + premise_db_dict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
                exchange.update({'database' : exchange_link['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchange_link = [ds['code'] for ef in biosphere_db_dict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchange_link)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    db_name = 'nitrous-oxide-ei310-' + premise_db_name
    if db_name in bd.databases:
        del bd.databases[db_name]
    wurst.write_brightway2_database(db_linked, db_name)
    print(premise_db_name + ' linking and writing successful!')

linking database Base-2050...
21 datasets
	191 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-Base-2050 (97 exchanges)
		nitrous-oxide-ei310-Base-2050 (51 exchanges)
		ecoinvent-3.10-biosphere (43 exchanges)
	0 unlinked exchanges (0 types)
		


100%|██████████| 21/21 [00:00<00:00, 750.27it/s]


10:39:40 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-Base-2050
Base-2050 linking and writing successful!
linking database RCP19-2050...
21 datasets
	191 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-RCP19-2050 (97 exchanges)
		nitrous-oxide-ei310-RCP19-2050 (51 exchanges)
		ecoinvent-3.10-biosphere (43 exchanges)
	0 unlinked exchanges (0 types)
		


100%|██████████| 21/21 [00:00<00:00, 841.17it/s]


10:41:05 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-RCP19-2050
RCP19-2050 linking and writing successful!
linking database RCP26-2050...
21 datasets
	191 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-RCP26-2050 (97 exchanges)
		nitrous-oxide-ei310-RCP26-2050 (51 exchanges)
		ecoinvent-3.10-biosphere (43 exchanges)
	0 unlinked exchanges (0 types)
		


100%|██████████| 21/21 [00:00<00:00, 841.29it/s]


10:42:35 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-RCP26-2050
RCP26-2050 linking and writing successful!
